In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pantab

In [2]:
# Input file paths
state_folder = "Virginia"
payer_folder = "Cigna Analysis"

# Contract Designation: Specify New contract date range
contract_start_date = '2025-01-01'
contract_end_date = '2025-12-31'

# find Excel files with "PowerBI" in the name
base_folder = Path(r"C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology\1. Payor Data & Contracts")
powerbi_folder = base_folder / state_folder / payer_folder / "PowerBI"
matching_files = [
    f for f in powerbi_folder.glob("*.xlsx")
    if "powerbi" in f.name.lower()
]

if not matching_files:
    raise FileNotFoundError(f'No Excel files with "PowerBI" in the name found in: {powerbi_folder}')

# get most recently modified file
latest_file = max(matching_files, key=lambda f: f.stat().st_mtime)

print(latest_file)

# read Excel file
billing_raw = pd.read_excel(latest_file)

C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology\1. Payor Data & Contracts\Virginia\Cigna Analysis\PowerBI\Cigna VA - 2024 & YTD Oct 2025 - as of 2.12.26 PowerBI.xlsx


In [3]:
# Bring in Service Category Mapping
service_map = pd.read_excel(r'C:\Users\pirat\Dropbox\Consulting Inc\Josh Perkins Data\CMS Data\Service Category Mapping.xlsx')
billing_clean = billing_raw.copy()

# Bring in Location Mapping
location_map = pd.read_excel(
    r"C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology"
    r"\1. Payor Data & Contracts\99-Tableau\ADCS Location Map.xlsx"
)

In [4]:
# "Old Contract" period will be exactly one year prior to "New Contract" period dynamically
old_contract_start_date = pd.to_datetime(contract_start_date) - pd.DateOffset(years=1)
old_contract_end_date = pd.to_datetime(contract_end_date) - pd.DateOffset(years=1)

In [5]:
# Initial cleanup

# if "Payer" is null delete row
billing_clean = billing_clean.dropna(subset=['Payer'])

# replace col header spaces with underscores
billing_clean.columns = billing_clean.columns.str.replace(' ', '_')
service_map.columns = service_map.columns.str.replace(' ', '_')

# if "Provider SubGrp 1" is anything other than "Physicians" or "Extenders", change to "Physicians"
billing_clean.loc[~billing_clean['Provider_SubGrp_1'].isin(['Physicians', 'Extenders']), 'Provider_SubGrp_1'] = 'Physicians'

# CPT Code columns based on Service Item
billing_clean['CPT_Code_Full'] = billing_clean['Service_Item'].str.split(' - ').str[0]
billing_clean['Service_Description'] = billing_clean['Service_Item'].str.split(' - ').str[1]
billing_clean['CPT_Code_Core'] = billing_clean['CPT_Code_Full'].str.split('-').str[0]

# join service map "Service Category" to billing_clean on "CPT_Code_Core" and "CPT Code"
billing_clean = billing_clean.merge(service_map[['Code_Full', 'Category_Name']], 
                                    left_on='CPT_Code_Full', right_on='Code_Full', how='left')

# rename "Category_Name" to "Service_Category"
billing_clean = billing_clean.rename(columns={'Category_Name': 'Service_Category'})

In [6]:
# Date cleanup

# if year is missing, delete row
billing_clean = billing_clean.dropna(subset=['Year'])

# change year from float to int
billing_clean['Year'] = billing_clean['Year'].astype(int)

# Create date from 'Year' and 'Date_-_Month' columns with day as 1
billing_clean['Date'] = pd.to_datetime(billing_clean['Year'].astype(str) + '-' + billing_clean['Date_-_Month'].astype(str) + '-01')

In [7]:
# Calculated Columns

# new col "Avg Alwd" == "Act_Alwd" / "Count"
billing_clean['Avg_Alwd'] = billing_clean['Act_Alwd'] / billing_clean['Count']

# new col "Total_Paid" == "Act_TP_Paid*" + "Pat_Paid"
billing_clean['Total_Paid'] = (
    pd.to_numeric(billing_clean['Act_TP_Paid*'], errors='coerce')
      .add(pd.to_numeric(billing_clean['Pat_Paid'], errors='coerce'), fill_value=0)
      .round(5)
)*-1

# new col "Avg_Paid" == "Total_Paid" / "Count"
billing_clean['Avg_Paid'] = billing_clean['Total_Paid'] / billing_clean['Count']

In [8]:
# Payor Mapping to LOB

# First make a series based on the raw insurance name
lob = billing_clean["Payer_Sub_Group_2"].fillna("").str.lower()

# build a guess for every row
payor_guess = np.select(
    [
        lob.str.contains("medicare") | lob.str.contains("mcr"),
        lob.str.contains("medicaid") | lob.str.contains("mcd"),
        lob.str.contains("tricare") | lob.str.contains("exchange"),
    ],
    ["MCR", "MCD", "Exchange"],
    default="Commercial",
)

# map the guesses back to the dataframe
billing_clean["Payor_LOB"] = payor_guess

# apply any overrides based on contract name
overrides = {
    # "AETNA Commercial FL Loc 99": "TEST"
}

billing_clean["Payor_LOB"] = (
    billing_clean["Contract_Name"].map(overrides)
    .fillna(billing_clean["Payor_LOB"])
)


In [9]:
# Add cols for commonly changed variables

# Identify if rows are in the new contract period (New, Old, or Neither)
billing_clean['Contract_Period'] = np.where(
    (billing_clean['Date'] >= pd.to_datetime(contract_start_date)) & (billing_clean['Date'] <= pd.to_datetime(contract_end_date)),
    'New Contract',
    np.where(
        (billing_clean['Date'] >= old_contract_start_date) & (billing_clean['Date'] <= old_contract_end_date),
        'Old Contract',
        'Neither'
    )
)


In [10]:
# merge location map to billing_clean
billing_clean = billing_clean.merge(
    location_map[["Location_Name", "Locality"]],
    how="left",
    on="Location_Name",
)

# print locations where Locality is missing
missing_locality = (
    billing_clean.loc[billing_clean["Locality"].isna(), "Location_Name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if missing_locality:
    print("Location_Name values missing Locality:")
    for loc in missing_locality:
        print(f"- {loc}")
else:
    print("All Location_Name values have a Locality.")

All Location_Name values have a Locality.


In [11]:
#re-order columns to have the new ones closer to the front
cols = billing_clean.columns.tolist()
new_order = ['CPT_Code_Full', 'CPT_Code_Core', 'Service_Description', 'Service_Category',
      'Payor_LOB', 'Date', 'Avg_Alwd', 'Total_Paid', 'Avg_Paid',  'Contract_Period', 'Locality'     
]
cols = new_order + [c for c in cols if c not in new_order]
billing_clean = billing_clean[cols]

In [12]:
#output file path
payer_name = payer_folder.replace(" Analysis", "")
output_dir = base_folder / state_folder / payer_folder / "Tableau"
base = output_dir / f"{state_folder}_{payer_name}_Billing_Cleaned"

exports = {
    "xlsx":  base.with_suffix(".xlsx"),
    "hyper": base.with_suffix(".hyper"),
}


# Make a Hyper-safe copy (mainly: booleans)
billing_hyper = billing_clean.copy().reset_index(drop=True)

bool_cols = billing_hyper.select_dtypes(include=["bool", "boolean"]).columns
if len(bool_cols):
    # keep nulls while avoiding Arrow bool type (works well with Hyper)
    billing_hyper[bool_cols] = billing_hyper[bool_cols].astype("Int8")  # or "string"

writers = {
    "xlsx":  lambda df, p: df.to_excel(p, index=False),
    "hyper": lambda df, p: pantab.frame_to_hyper(df, str(p), table="Extract"),
}

for fmt, path in exports.items():
    df_to_write = billing_hyper if fmt == "hyper" else billing_clean
    writers[fmt](df_to_write, path)
